# v1a.1 Dual-Kidney Prompt Generator
Canonical RAS orientation, separate left/right tracks, local size/offset heads, overfit gate, full training, expansion sweep, and frozen MedSAM evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PACKAGE_CANDIDATES = [
    Path('/content/v1a1_dual_kidney_pipeline.zip'),
    Path('/content/drive/MyDrive/medsam-kits23/v1a1_dual_kidney_pipeline.zip'),
]
PACKAGE = next((p for p in PACKAGE_CANDIDATES if p.exists()), None)
if PACKAGE is None:
    raise FileNotFoundError('Upload v1a1_dual_kidney_pipeline.zip to /content or MyDrive/medsam-kits23')
print('Package:', PACKAGE)


In [ ]:
!rm -rf /content/v1a1_dual_kidney_pipeline
!unzip -q "{PACKAGE}" -d /content
%cd /content/v1a1_dual_kidney_pipeline
!pip install -q -r requirements.txt


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## Discover the archive and the MedSAM checkpoint


In [ ]:
!python scripts/discover_drive_assets.py \
  --drive-root /content/drive/MyDrive \
  --project-dir /content/drive/MyDrive/medsam-kits23 \
  --output-json /content/v1a1_assets.json


## Stage 1 — preprocess/cache, orientation audit, dual manifest, sanity image


In [ ]:
!python run_pipeline.py \
  --drive-root /content/drive/MyDrive \
  --project-dir /content/drive/MyDrive/medsam-kits23 \
  --work-root /content/v1a1_work \
  --stop-after sanity


In [ ]:
from IPython.display import Image, display
display(Image('/content/v1a1_work/manifest/dual_target_sanity.png'))


## Stage 2 — deterministic overfit gate


In [ ]:
!python scripts/overfit_dual.py \
  --manifest /content/v1a1_work/manifest/v1a1_dual_manifest.csv \
  --output-dir /content/v1a1_work/overfit \
  --positive-slices 48 \
  --negative-slices 12 \
  --steps 1200 \
  --batch-size 12 \
  --base-channels 24 \
  --expansion-scale 1.15 \
  --device cuda


In [ ]:
display(Image('/content/v1a1_work/overfit/overfit_predictions.png'))


## Stage 3 — full training with per-epoch Drive backup


In [ ]:
!python scripts/train_dual.py \
  --manifest /content/v1a1_work/manifest/v1a1_dual_manifest.csv \
  --output-dir /content/v1a1_work/run \
  --backup-dir /content/drive/MyDrive/medsam-kits23/v1a1_results/run \
  --epochs 40 \
  --batch-size 32 \
  --num-workers 2 \
  --input-size 256 \
  --heatmap-size 64 \
  --input-mode 2.5d \
  --base-channels 24 \
  --lr 3e-4 \
  --warmup-epochs 3 \
  --metric-expansion-scale 1.15 \
  --early-stopping-patience 10 \
  --device cuda


## Stage 4 — export validation and test predictions


In [ ]:
CHECKPOINT='/content/v1a1_work/run/best_continuous.pt'

!python scripts/predict_dual.py \
  --manifest /content/v1a1_work/manifest/v1a1_dual_manifest.csv \
  --checkpoint $CHECKPOINT \
  --output-csv /content/v1a1_work/run/val_predictions.csv \
  --split val --device cuda

!python scripts/predict_dual.py \
  --manifest /content/v1a1_work/manifest/v1a1_dual_manifest.csv \
  --checkpoint $CHECKPOINT \
  --output-csv /content/v1a1_work/run/test_predictions.csv \
  --split test --device cuda


## Stage 5 — validation expansion sweep


In [ ]:
!python scripts/sweep_expansion.py \
  --predictions /content/v1a1_work/run/val_predictions.csv \
  --output-dir /content/v1a1_work/run/val_expansion_sweep \
  --scales 1.10,1.15,1.20,1.25


## Stage 6 — prompt evaluation


In [ ]:
!python scripts/evaluate_dual_prompts.py \
  --predictions /content/v1a1_work/run/test_predictions.csv \
  --output-dir /content/v1a1_work/run/test_prompt_evaluation \
  --variant both \
  --expansion-scale 1.15


## Optional — frozen MedSAM pilot


In [ ]:
import json
assets=json.load(open('/content/v1a1_assets.json'))
MEDSAM_WEIGHT=assets['medsam_checkpoint']
print(MEDSAM_WEIGHT)


In [ ]:
!python scripts/setup_medsam.py \
  --repo-dir /content/v1a1_work/MedSAM \
  --checkpoint-source "{MEDSAM_WEIGHT}" \
  --checkpoint-dest /content/v1a1_work/medsam_weights/medsam_vit_b.pth

!python scripts/evaluate_medsam_dual.py \
  --predictions /content/v1a1_work/run/test_predictions.csv \
  --medsam-repo /content/v1a1_work/MedSAM \
  --checkpoint /content/v1a1_work/medsam_weights/medsam_vit_b.pth \
  --output-dir /content/v1a1_work/run/medsam_test_pilot \
  --expansion-scale 1.15 \
  --max-samples 100 \
  --device cuda
